<a href="https://colab.research.google.com/github/hamzakhattak-AIDev/flyrank-ML-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hamzakhattak-AIDev/flyrank-ML-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

# 1. Two paper findings + my methodology questions

### Finding 1: Search Traffic Decay Models Achieve High Generalizable Accuracy Across Domains

* **Methodology Question (Label Origin):** How is the exact cutoff for "decay" defined across heterogeneous client verticals? If traffic baseline thresholds vary across niche industries (e.g., high-volume consumer e-commerce vs. low-volume enterprise B2B), does a fixed percentage drop label introduce systemic misclassification for seasonal traffic patterns?
* **Validation Design Check:** Was the model evaluated on held-out client domains or held-out time periods? Evaluating traffic decay models on randomly split URLs across shared domains creates domain leakage, as technical SEO configurations and client-level authority signals are exposed to both training and test sets.

---

### Finding 2: Content Refresh Timeliness Drives SERP Rank Recovery

* **Methodology Question (Causal Inference vs. Correlation):** Does the validation framework control for search engine algorithm updates, seasonality, or secondary promotional campaigns that coincide with content refreshes?
* **Validation Design Check:** Are refresh outcome labels measured over a uniform post-intervention window, or does variable tracking duration bias the measured recovery rate toward high-authority domains?

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load raw dataset directly from repository
raw_url = "https://raw.githubusercontent.com/hamzakhattak-AIDev/flyrank-ML-internship/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(raw_url)

# Target formulation: 1 if declining trend, 0 otherwise
df["target"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Historical pre-period feature set
feature_cols = [
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "clicks_90d"
]

print(f"Dataset shape: {df.shape}")
print(f"Total Unique Clients: {df['client_id'].nunique()}")
print(f"Overall Class Distribution:\n{df['target'].value_counts(normalize=True)}")

Dataset shape: (30000, 45)
Total Unique Clients: 32
Overall Class Distribution:
target
1    0.542067
0    0.457933
Name: proportion, dtype: float64


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

# 2. My model under an honest split (before/after)

### Split Design Audit: Naive Random Split vs. Honest Grouped Client Split

| Metric | Naive Random Split (Leaky) | Honest Grouped Split (Client Hold-Out) | Optimism Gap (Random - Grouped) |
| :--- | :---: | :---: | :---: |
| **Accuracy** | 65.18% | **59.14%** | +6.04% |
| **Precision** | 63.69% | **56.57%** | +7.12% |
| **Recall** | 83.21% | **86.31%** | -3.10% |
| **F1 Score** | 0.7215 | **0.6834** | +0.0381 |
| **ROC-AUC** | 0.7007 | **0.6291** | **+0.0715** |

### Key Findings & Optimism Bias Audit
* **Overestimating Generalizability:** The naive random split yielded an inflated ROC-AUC of **0.7007**, whereas evaluating on completely unseen clients dropped ROC-AUC to **0.6291** (an optimism gap of +0.0715).
* **Intra-Client Data Leakage:** Random splitting allows URLs from the same client domain to exist in both training and test sets. The model memorizes client-specific traffic scale and position baselines rather than learning generalizable organic decay mechanics.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

X = df[feature_cols]
y = df["target"]
groups = df["client_id"]

# 1. NAIVE RANDOM SPLIT (Leaky - allows same client across train/test)
X_train_rnd, X_test_rnd, y_train_rnd, y_test_rnd = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

rf_random = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_random.fit(X_train_rnd, y_train_rnd)

y_pred_rnd = rf_random.predict(X_test_rnd)
y_prob_rnd = rf_random.predict_proba(X_test_rnd)[:, 1]

# 2. HONEST GROUPED SPLIT (Zero client overlap)
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=groups))

train_df = df.iloc[train_idx]
test_df = df.iloc[test_idx]

rf_grouped = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_grouped.fit(train_df[feature_cols], train_df["target"])

y_pred_grp = rf_grouped.predict(test_df[feature_cols])
y_prob_grp = rf_grouped.predict_proba(test_df[feature_cols])[:, 1]

# 3. COMPARISON TABLE
audit_comp = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "Naive Random Split (Leaky)": [
        accuracy_score(y_test_rnd, y_pred_rnd),
        precision_score(y_test_rnd, y_pred_rnd, zero_division=0),
        recall_score(y_test_rnd, y_pred_rnd, zero_division=0),
        f1_score(y_test_rnd, y_pred_rnd, zero_division=0),
        roc_auc_score(y_test_rnd, y_prob_rnd)
    ],
    "Honest Grouped Split (Client Hold-out)": [
        accuracy_score(test_df["target"], y_pred_grp),
        precision_score(test_df["target"], y_pred_grp, zero_division=0),
        recall_score(test_df["target"], y_pred_grp, zero_division=0),
        f1_score(test_df["target"], y_pred_grp, zero_division=0),
        roc_auc_score(test_df["target"], y_prob_grp)
    ]
})

audit_comp["Optimism Gap (Random - Grouped)"] = audit_comp["Naive Random Split (Leaky)"] - audit_comp["Honest Grouped Split (Client Hold-out)"]

print("=== SPLIT DESIGN AUDIT: BEFORE VS AFTER ===")
print(audit_comp.to_string(index=False))

=== SPLIT DESIGN AUDIT: BEFORE VS AFTER ===
   Metric  Naive Random Split (Leaky)  Honest Grouped Split (Client Hold-out)  Optimism Gap (Random - Grouped)
 Accuracy                    0.651833                                0.591433                         0.060401
Precision                    0.636856                                0.565661                         0.071195
   Recall                    0.832103                                0.863131                        -0.031028
 F1 Score                    0.721504                                0.683430                         0.038074
  ROC-AUC                    0.700654                                0.629118                         0.071536


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

# 3. Leakage audit

### Feature Set Audit Results
Evaluated feature matrix ($X$): `['days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'clicks_90d']`

* **Target Proxies:** No direct target definitions or trend direction metrics (`trend_direction`) exist within $X$.
* **Post-Period Signals:** All 5 features are strictly derived from pre-intervention historical windows (last 90 days), ensuring zero lookahead leakage into post-evaluation outcomes.
* **Conclusion:** The feature set is verified clean. The primary source of performance degradation observed in Section 2 stems purely from validation split design (domain leakage), not feature leakage.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Programmatic Leakage Inspection
forbidden_tokens = ["trend", "post", "future", "target", "label", "outcome"]

detected_leakage = [col for col in feature_cols if any(token in col.lower() for token in forbidden_tokens)]

print("=== FEATURE LEAKAGE AUDIT ===")
print(f"Features Evaluated ({len(feature_cols)}): {feature_cols}")
if not detected_leakage:
    print("LEAKAGE CHECK PASSED: Zero target proxies or post-period signals detected in the feature set.")
else:
    print(f"LEAKAGE DETECTED: Remove forbidden feature(s): {detected_leakage}")

=== FEATURE LEAKAGE AUDIT ===
Features Evaluated (5): ['days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'clicks_90d']
LEAKAGE CHECK PASSED: Zero target proxies or post-period signals detected in the feature set.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

# 4. Claim rewrite

### Error Example Insights
* **False Positives ($n = 2,087$):** Content with low recent impression volume (e.g., 307 impressions at rank 39.8) is flagged as decaying due to poor rank metrics, despite holding stable traffic baselines.
* **False Negatives ($n = 431$):** Recently updated pages (e.g., 4 days since update) with low impression counts are misclassified as healthy, masking early organic decline.

---

### Claim Rewrites (Using Decision-Support Safe Language)

* **Overconfident Claim:** *"Our machine learning model accurately predicts organic search traffic decay across all client websites with over 72% F1 score."*
* **Audited Decision-Support Claim:** *"Under a client-grouped holdout evaluation, the Random Forest model **observed** an F1 score of **0.6834** and an ROC-AUC of **0.6291**. The model functions as a **directional decision-support tool**, successfully prioritizing **86.31%** of decaying content for editorial review while demonstrating reduced accuracy on newly onboarded domains with atypical impression distributions."*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Extract False Positives (FP) and False Negatives (FN) on Grouped Test Split
test_eval = test_df.copy()
test_eval["pred_class"] = y_pred_grp
test_eval["pred_prob"] = y_prob_grp

fp_df = test_eval[(test_eval["target"] == 0) & (test_eval["pred_class"] == 1)]
fn_df = test_eval[(test_eval["target"] == 1) & (test_eval["pred_class"] == 0)]

print("=== ERROR SUMMARY ===")
print(f"Total Test Instances: {len(test_eval)}")
print(f"False Positives (Predicted Decay, Actually Stable): {len(fp_df)}")
print(f"False Negatives (Predicted Stable, Actually Decaying): {len(fn_df)}")

print("\n--- False Positive Examples (Predicted Decay, Actually Stable) ---")
print(fp_df[["client_id", "days_since_last_update", "impressions_90d", "avg_position", "ctr", "pred_prob"]].head(3).to_string(index=False))

print("\n--- False Negative Examples (Predicted Stable, Actually Decaying) ---")
print(fn_df[["client_id", "days_since_last_update", "impressions_90d", "avg_position", "ctr", "pred_prob"]].head(3).to_string(index=False))

=== ERROR SUMMARY ===
Total Test Instances: 6163
False Positives (Predicted Decay, Actually Stable): 2087
False Negatives (Predicted Stable, Actually Decaying): 431

--- False Positive Examples (Predicted Decay, Actually Stable) ---
        client_id  days_since_last_update  impressions_90d  avg_position  ctr  pred_prob
client_8527a891e2                     103              307          39.8 0.00   0.592697
client_4e07408562                      13             2426          30.0 0.12   0.670093
client_f369cb89fc                      20              371           5.4 1.35   0.533204

--- False Negative Examples (Predicted Stable, Actually Decaying) ---
        client_id  days_since_last_update  impressions_90d  avg_position  ctr  pred_prob
client_8527a891e2                     104                4          36.3  0.0   0.327890
client_f369cb89fc                       8                2           7.5  0.0   0.313521
client_f369cb89fc                      20               25           6.2 

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.